In [0]:
import pyspark
from pyspark.sql.functions import regexp_extract,col,when

In [0]:
df = spark.read.option("header", True).option("inferSchema", True).csv("/FileStore/tables/ipl_commentary_data-3.csv")

In [0]:
df_extracted = df.withColumn("Bowler", regexp_extract("ball_commentary", r"^(.*?) to ", 1)) \
                 .withColumn("Batsman", regexp_extract("ball_commentary", r" to (.*?),", 1)) \
                 .withColumn("raw_runs", regexp_extract("ball_commentary",r",\s*(FOUR|SIX|no run|1 run|2 runs|3 runs|leg bye|bye|out|wicket)", 1)) \
                 .withColumn("Runs", when(df.ball_commentary.rlike("FOUR"), 4)
                                      .when(df.ball_commentary.rlike("SIX"), 6)
                                      .when(df.ball_commentary.rlike(r"\b1 run\b"), 1)
                                      .when(df.ball_commentary.rlike(r"\b2 runs\b"), 2)
                                      .when(df.ball_commentary.rlike(r"\b3 runs\b"), 3)
                                      .when(df.ball_commentary.rlike("no run"), 0)
                                      .when(df.ball_commentary.rlike("OUT"), 0)
                                      .otherwise(None))\
                                    

df_extracted.select("ball_no", "over_no", "ball_commentary", "bowler", "batsman", "runs").display()


ball_no,over_no,ball_commentary,bowler,batsman,runs
1,0.1,"Siraj to Rohit, 2 runs, straightaway into the pads. Siraj pitches it up but doesn't find much swing, leaning into middle and leg stumps and is clipped away for a couple through square leg. Rohit's away",Siraj,Rohit,2
2,0.2,"Siraj to Rohit, no run, full ball on middle, defended watchfully down the pitch",Siraj,Rohit,0
3,0.3,"Siraj to Rohit, no run, indications of the pace of this surface, and the bounce. Back of a length, probably wasn't there to pull in any case. Rohit goes for it but the ball sneaks under him and strikes him on the thigh",Siraj,Rohit,0
4,0.4,"Siraj to Rohit, 2 runs, short of length around the hip, Rohit whips it wide of fine leg, gets back for two",Siraj,Rohit,2
5,0.5,"Siraj to Rohit, no run, short of length delivery on off, Rohit is right behind the line in defence",Siraj,Rohit,0
6,0.6,"Siraj to Rohit, 1 run, back of a length delivery on off, Rohit tucks it through to deep square leg where a fielder was brought in at some point during the over",Siraj,Rohit,1
7,1.1,"Jamieson to Rohit, 1 run, Jamieson introduces himself to the IPL with an in-ducker. Really full, bending in on middle, Rohit digs it out wide of mid-on",Jamieson,Rohit,1
8,1.2,"Jamieson to Chris Lynn, no run, back of a length on leg, Lynn is struck on the thigh pad looking to pull it away",Jamieson,Chris Lynn,0
9,1.3,"Jamieson to Chris Lynn, no run, pitched up around off, Lynn drives to mid-off. The ball doesn't seem to be coming on all that well, at least that's the early impression",Jamieson,Chris Lynn,0
10,1.4,"Jamieson to Chris Lynn, no run, an edge that falls short of first slip. Clearly the lack of pace in the pitch on display here. Lynn went hard on the drive as Jamieson pitched it up, genuine edge, but on the bounce to the fielder diving to his left",Jamieson,Chris Lynn,0
